# 원본 이미지 분포 비교와 로컬 GPU 스모크 테스트
train/test는 관찰 목적으로만 비교하며 test 이미지는 학습·분할 생성에 사용하지 않습니다.
분위수 겹침은 분포 동일성의 증명이 아닙니다. 방향은 축의 180° 주기를 고려해
수평선과의 최소 각도(0~90°)로 비교하며 **70° 이상** 비율을 표시합니다.
사진의 방향은 배경 직선의 기울기 추정치이며 실제 카메라 회전의 정답이 아닙니다.


In [ ]:
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 저장소 루트 또는 notebooks에서 실행합니다.
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "config.py").is_file():
    raise RuntimeError("커널 작업 경로를 저장소 루트 또는 notebooks로 설정하세요.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["WANDB_MODE"] = "disabled"
from src.config import PROJECT_ROOT, get_device  # noqa: E402

SEED = 42
BORDER = 20
ANGLE_THRESHOLD = 70.0
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
paths = {
    group: sorted(
        p
        for p in (PROJECT_ROOT / "data" / "raw" / folder).rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )
    for group, folder in {"train": "train_images", "test": "test_images"}.items()
}
for group, items in paths.items():
    print(f"{group}: {len(items):,}장")
    if not items:
        raise FileNotFoundError(f"{group} 이미지가 없습니다.")
print("로컬 장치:", get_device())

In [ ]:
def read_image(path: Path) -> np.ndarray:
    """한글 경로를 포함한 이미지 파일을 BGR로 읽습니다."""
    image = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"이미지를 읽을 수 없습니다: {path}")
    return image


def border_mask(shape: tuple, width: int = BORDER) -> np.ndarray:
    """모서리를 중복 집계하지 않는 테두리 마스크를 만듭니다."""
    height, image_width = shape[:2]
    if min(height, image_width) <= 2 * width:
        raise ValueError("이미지가 테두리 폭보다 작습니다.")
    mask = np.ones((height, image_width), dtype=bool)
    mask[width:-width, width:-width] = False
    return mask


rng = np.random.default_rng(SEED)
selected = rng.choice(len(paths["test"]), size=min(60, len(paths["test"])), replace=False)
fig, axes = plt.subplots(10, 6, figsize=(18, 28))
for axis in axes.flat:
    axis.axis("off")
for axis, index in zip(axes.flat, selected, strict=False):
    path = paths["test"][index]
    image = read_image(path)
    thumbnail = cv2.resize(image, (320, max(1, round(image.shape[0] * 320 / image.shape[1]))))
    axis.imshow(cv2.cvtColor(thumbnail, cv2.COLOR_BGR2RGB))
    axis.set_title(path.name, fontsize=7)
fig.suptitle("Test: seed=42, 60 images")
plt.tight_layout()
plt.show()

## 색 분포
S·V는 0~1, 배경 RGB는 0~255입니다. 테두리 20px의 평균을 이미지별로 계산하므로
큰 이미지가 통계를 지배하지 않습니다. 배경에 알약·그림자가 있으면 배경색 추정도 영향을 받습니다.
H는 원형 평균을 사용하며, 무채색에서 의미가 약하므로 색상 집중도도 함께 기록합니다.


In [ ]:
def color_features(image: np.ndarray) -> dict:
    """이미지 HSV와 원본 해상도 테두리의 색 통계를 계산합니다."""
    hsv = cv2.cvtColor(image.astype(np.float32) / 255, cv2.COLOR_BGR2HSV)
    hue = np.deg2rad(hsv[..., 0])
    weights = hsv[..., 1]
    vector = np.sum(weights * np.exp(1j * hue)) / max(float(weights.sum()), 1e-9)
    edge = border_mask(image.shape)
    background_rgb = image[edge][:, ::-1].mean(axis=0)
    return {
        "hue_deg": float(np.rad2deg(np.angle(vector)) % 360),
        "hue_concentration": float(abs(vector)),
        "saturation": float(hsv[..., 1].mean()),
        "value": float(hsv[..., 2].mean()),
        "background_s": float(hsv[..., 1][edge].mean()),
        "background_v": float(hsv[..., 2][edge].mean()),
        **dict(zip(("background_r", "background_g", "background_b"), background_rgb, strict=True)),
    }

## 방향 추정과 검수
배경색과의 차이로 전경을 분리하고 타원 장축을 추정합니다. 장단축 비율 1.3 미만,
작거나 경계에 닿은 윤곽은 제외합니다. 붙은 알약·각인·그림자를 오인할 수 있어 자동 측정은 후보입니다.
사진 방향은 바깥 20% 영역에서 검출된 긴 직선 중 서로 비슷한 선들의 방향만 사용합니다.
기준선이 없거나 합의가 약하면 결측값으로 둡니다. 90° 회전 및 카메라 방향을 확정하지 못합니다.


In [ ]:
def direction_features(image: np.ndarray) -> tuple[list, float, list, np.ndarray]:
    """알약 장축 후보와 배경 기준선 기울기를 추정하고 검수 자료를 반환합니다."""
    scale = min(1.0, 900 / max(image.shape[:2]))
    small = cv2.resize(image, None, fx=scale, fy=scale)
    height, width = small.shape[:2]
    lab = cv2.cvtColor(small, cv2.COLOR_BGR2LAB).astype(float)
    background = np.median(lab[border_mask(small.shape, max(1, round(BORDER * scale)))], axis=0)
    distance = np.linalg.norm(lab - background, axis=2)
    values = np.clip(distance, 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(values, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    pills = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if len(contour) < 5 or not 0.0005 * height * width < area < 0.15 * height * width:
            continue
        x, y, box_width, box_height = cv2.boundingRect(contour)
        if x <= 1 or y <= 1 or x + box_width >= width - 1 or y + box_height >= height - 1:
            continue
        ellipse = cv2.fitEllipse(contour)
        center, (axis_a, axis_b), angle = ellipse
        if min(axis_a, axis_b) <= 0 or max(axis_a, axis_b) / min(axis_a, axis_b) < 1.3:
            continue
        angle = (angle + (90 if axis_b > axis_a else 0)) % 180
        tilt = min(angle, 180 - angle)
        pills.append((tilt, center, max(axis_a, axis_b), angle))

    edges = cv2.Canny(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), 70, 160)
    edges[height // 5 : 4 * height // 5, width // 5 : 4 * width // 5] = 0
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=60,
        minLineLength=max(30, min(height, width) // 4),
        maxLineGap=15,
    )
    candidates = []
    if lines is not None:
        # OpenCV 버전별 (N, 1, 4), (N, 4) 반환 형태를 통일합니다.
        for x1, y1, x2, y2 in np.asarray(lines).reshape(-1, 4):
            angle = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
            candidates.append((angle, np.hypot(x2 - x1, y2 - y1), (x1, y1, x2, y2)))
    photo_angle = np.nan
    accepted = []
    if len(candidates) >= 2:
        angles = np.array([item[0] for item in candidates])
        lengths = np.array([item[1] for item in candidates])
        differences = abs(angles[:, None] - angles[None, :])
        support = np.minimum(differences, 180 - differences) <= 5
        cluster = support[np.argmax(support @ lengths)]
        if cluster.sum() >= 2 and lengths[cluster].sum() / lengths.sum() >= 0.7:
            vector = np.sum(lengths[cluster] * np.exp(2j * np.deg2rad(angles[cluster])))
            angle = (np.rad2deg(np.angle(vector)) / 2) % 180
            photo_angle = min(angle, 180 - angle)
            accepted = [item[2] for item, keep in zip(candidates, cluster, strict=True) if keep]
    return pills, photo_angle, accepted, small


records, pill_records, failures = [], [], []
for group, items in paths.items():
    for index, path in enumerate(items):
        try:
            image = read_image(path)
            pills, photo_angle, _, _ = direction_features(image)
            records.append(
                {
                    "group": group,
                    "path": str(path),
                    **color_features(image),
                    "photo_tilt": photo_angle,
                    "pill_count": len(pills),
                    "pill_max_tilt": max((p[0] for p in pills), default=np.nan),
                }
            )
            pill_records.extend(
                {"group": group, "path": str(path), "pill_tilt": p[0]} for p in pills
            )
        except (ValueError, cv2.error) as error:
            failures.append({"path": str(path), "error": str(error)})
        if (index + 1) % 100 == 0:
            print(f"{group}: {index + 1}/{len(items)}")
features = pd.DataFrame(records)
pill_features = pd.DataFrame(pill_records, columns=["group", "path", "pill_tilt"])
display(pd.DataFrame(failures))
if features.empty:
    raise RuntimeError("분석에 성공한 이미지가 없습니다.")

### 밝기·채도·배경색: “같다 / 이렇게 다르다” 해석
아래 표와 히스토그램을 보고 train과 test를 비교합니다. 아직 결과를 판정한 문장은 아닙니다.

| 항목 | 같다에 가까운 경우 | 이렇게 다른 경우 |
|---|---|---|
| 밝기(V, `value`) | 5·50·95% 분위수가 가깝고 히스토그램도 비슷하다 | test의 분위수가 낮으면 더 어둡고, 높으면 더 밝다. 양 끝이 더 벌어지면 밝기 범위가 더 넓다 |
| 채도(S, `saturation`) | 분위수와 히스토그램이 비슷하다 | test의 분위수가 낮으면 색이 더 옅고, 높으면 더 선명하다 |
| 배경색(`background_r/g/b`, `background_s/v`) | 테두리 20px의 RGB·S·V 분위수와 히스토그램이 비슷하다 | test의 배경이 더 밝거나 어둡다, 채도가 높거나 낮다, RGB 구성 차이가 있다는 식으로 설명한다 |

`test_minus_train_median`은 양수면 test의 중앙값이 높고 음수면 낮다는 뜻입니다.
`central90_overlap=True`는 5~95% 구간이 일부 겹친다는 뜻이며, 이것만으로 “같다”로 판정하지 않습니다.
`overlap_over_union`과 히스토그램을 함께 봅니다. “가깝다”의 수치 허용범위는 아직 정하지 않았습니다.
배경 RGB는 채널별 주변 분포이므로 특정 배경색이 더 많다는 결론은 그리드 이미지도 함께 확인합니다.


In [ ]:
metrics = [
    "saturation",
    "value",
    "background_s",
    "background_v",
    "background_r",
    "background_g",
    "background_b",
]
quantiles = features.groupby("group")[metrics].quantile([0.05, 0.5, 0.95])
display(quantiles)
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for axis, metric in zip(axes.flat, [*metrics, "hue_deg", "photo_tilt"], strict=True):
    values = features[metric].dropna()
    if values.empty:
        axis.set_title(f"{metric}: unavailable")
        continue
    bins = (
        np.linspace(0, 90, 19)
        if metric == "photo_tilt"
        else np.histogram_bin_edges(values, bins=25)
    )
    for group in paths:
        subset = features.loc[features.group == group, metric].dropna()
        if not subset.empty:
            axis.hist(
                subset,
                bins=bins,
                weights=np.ones(len(subset)) / len(subset),
                alpha=0.45,
                label=group,
            )
    axis.set_title(metric)
    axis.set_ylabel("Image proportion")
    axis.legend()
plt.tight_layout()
plt.show()

# 중앙 90% 구간의 교집합과 중앙값 차이를 기록합니다.
comparison = []
for metric in metrics:
    train_values = quantiles.loc["train", metric]
    test_values = quantiles.loc["test", metric]
    low = max(train_values.loc[0.05], test_values.loc[0.05])
    high = min(train_values.loc[0.95], test_values.loc[0.95])
    union = max(train_values.loc[0.95], test_values.loc[0.95]) - min(
        train_values.loc[0.05], test_values.loc[0.05]
    )
    comparison.append(
        {
            "metric": metric,
            "central90_overlap": high >= low,
            "overlap_over_union": max(0, high - low) / union if union else 1.0,
            "test_minus_train_median": test_values.loc[0.5] - train_values.loc[0.5],
        }
    )
display(pd.DataFrame(comparison))

### 알약 방향·사진 방향: “같다 / 이렇게 다르다” 해석
아래 각도 통계와 검수 이미지를 함께 봅니다. 70°는 뒤집힘이 아니라 수평선에서 크게 기울어진 상태입니다.

| 항목 | 같다에 가까운 경우 | 이렇게 다른 경우 |
|---|---|---|
| 알약 방향 | 알약 각도 히스토그램과 `pill_max_tilt`의 70° 이상 사진 비율이 비슷하다 | test의 비율이 더 높으면 크게 기울어진 알약이 있는 사진이 더 많다. 각도 히스토그램이 넓으면 방향이 더 다양하다 |
| 사진 방향 | 신뢰할 수 있는 배경 기준선이 검출된 사진에서 기울기 분포와 70° 이상 비율이 비슷하다 | test의 비율이 더 높으면 검출된 배경 기준선이 크게 기울어진 사진이 더 많다. 사진 전체의 실제 회전이라고 확정하지 않는다 |

`ratio_ge_70_valid`의 분모는 각도를 측정할 수 있는 사진 수입니다.
알약 기준은 한 사진에서 측정된 알약 중 하나라도 70° 이상이면 해당 사진을 셉니다.
원형·윤곽 불확실 알약과 배경 기준선이 없는 사진은 측정에서 제외됩니다.
`coverage`가 낮거나 train/test 사이에 크게 다르면 “같다 / 다르다”를 보류하고 **측정 불충분**으로 설명합니다.
각도 분위수는 보조 자료이며, 장축·기준선 표시가 실제 대상을 잘 따르는지 검수한 뒤 해석합니다.


In [ ]:
angle_summary = []
for group in paths:
    subset = features[features.group == group]
    for metric in ("pill_max_tilt", "photo_tilt"):
        valid = subset[metric].dropna()
        angle_summary.append(
            {
                "group": group,
                "metric": metric,
                "total_images": len(subset),
                "measurable_images": len(valid),
                "coverage": len(valid) / len(subset) if len(subset) else np.nan,
                "ratio_ge_70_valid": (valid >= ANGLE_THRESHOLD).mean(),
                "q05": valid.quantile(0.05),
                "q50": valid.median(),
                "q95": valid.quantile(0.95),
            }
        )
display(pd.DataFrame(angle_summary))
print("알약 기준 사진 비율: 측정 가능한 알약 중 하나라도 70° 이상인 사진 / 측정 가능한 사진")
print("결측 사진을 각도 0으로 취급하지 않습니다. 사진 기울기 추정에는 수동 검수가 필요합니다.")
fig, axis = plt.subplots(figsize=(9, 4))
for group in paths:
    values = pill_features.loc[pill_features.group == group, "pill_tilt"]
    if len(values):
        axis.hist(
            values,
            bins=np.arange(0, 95, 5),
            weights=np.ones(len(values)) / len(values),
            alpha=0.45,
            label=group,
        )
axis.set(xlabel="Pill axis tilt (degrees)", ylabel="Pill proportion")
axis.legend()
plt.show()

# 각 그룹의 일반 사례와 70° 이상 후보를 함께 검수합니다.
review = pd.concat(
    [
        features.groupby("group", group_keys=False).head(3),
        features[features.pill_max_tilt >= ANGLE_THRESHOLD]
        .groupby("group", group_keys=False)
        .head(2),
        features[features.photo_tilt.notna()].groupby("group", group_keys=False).head(2),
    ]
).drop_duplicates("path")
if len(review):
    fig, axes = plt.subplots(len(review), 1, figsize=(10, 5 * len(review)), squeeze=False)
    for axis, row in zip(axes.flat, review.itertuples(), strict=True):
        pills, photo, lines, image = direction_features(read_image(Path(row.path)))
        for _, center, length, angle in pills:
            delta = np.array([np.cos(np.deg2rad(angle)), np.sin(np.deg2rad(angle))]) * length / 2
            start = tuple(np.rint(np.array(center) - delta).astype(int))
            end = tuple(np.rint(np.array(center) + delta).astype(int))
            cv2.line(image, start, end, (0, 255, 0), 2)
        for x1, y1, x2, y2 in lines:
            cv2.line(image, (x1, y1), (x2, y2), (255, 0, 255), 2)
        axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{row.group}: {Path(row.path).name}, photo={photo:.1f}, pills={len(pills)}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()

## 승인한 로컬 GPU 스모크 설정
YOLO11m 사전학습 가중치 `yolo11m.pt`, MuSGD(옵티마이저), v1,
setup30fix_v1_s42 / fold 0, epochs=5, batch=16, imgsz=640, fraction=0.1,
seed=42, lr0=0.01, aug=on, mAP75-95를 사용합니다.
증강은 기존 AUG_PRESETS["on"]을 사용하며 분석 결과로 자동 변경하지 않습니다.
5 epoch 결과는 성능 비교 근거가 아닙니다. `close_mosaic=10` 기본값은 유지합니다.
아래 셀은 준비 상태와 실행 명령만 출력합니다. 파이프라인과 학습은 별도 실행 스위치로 켭니다.
실행 시 processed·runs·가중치·캐시 파일이 생성될 수 있습니다. 이번 코드 작성에서는 실행하지 않습니다.


In [ ]:
import inspect
import shlex
import subprocess

import torch
from ultralytics.engine.trainer import BaseTrainer

from data_process.data_yaml import data_yaml_path
from models.train import AUG_PRESETS, OPTIMIZERS

SMOKE = {
    "model_name": "yolo11m.pt",
    "label_version": "v1",
    "split_name": "setup30fix_v1_s42",
    "fold": 0,
    "epochs": 5,
    "batch": 16,
    "imgsz": 640,
    "fraction": 0.1,
    "seed": SEED,
    "optimizer": "MuSGD",
    "lr0": 0.01,
    "aug": "on",
}
EXP = "smoke_yolo11m_musgd_5ep"
print(pd.Series(SMOKE).to_string())
print("증강:", AUG_PRESETS[SMOKE["aug"]])
print("장치:", get_device())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(
    "MuSGD 지원:",
    "MuSGD" in OPTIMIZERS and "MuSGD" in inspect.getsource(BaseTrainer.build_optimizer),
)
print("data.yaml:", data_yaml_path(SMOKE["label_version"], SMOKE["split_name"], SMOKE["fold"]))
pipeline_command = [
    "uv",
    "run",
    "python",
    "-m",
    "data_process.data_main",
    "--label",
    SMOKE["label_version"],
    "--split",
    SMOKE["split_name"],
]
train_command = [
    "uv",
    "run",
    "python",
    "-m",
    "models.model_main",
    "--model",
    SMOKE["model_name"],
    "--label",
    SMOKE["label_version"],
    "--split",
    SMOKE["split_name"],
    "--folds",
    str(SMOKE["fold"]),
    "--epochs",
    str(SMOKE["epochs"]),
    "--batch",
    str(SMOKE["batch"]),
    "--imgsz",
    str(SMOKE["imgsz"]),
    "--fraction",
    str(SMOKE["fraction"]),
    "--seed",
    str(SMOKE["seed"]),
    "--optimizer",
    SMOKE["optimizer"],
    "--lr0",
    str(SMOKE["lr0"]),
    "--aug",
    SMOKE["aug"],
    "--exp",
    EXP,
]
print(shlex.join(pipeline_command))
print("WANDB_MODE=disabled " + shlex.join(train_command))

In [ ]:
# 직접 실행할 때만 True로 변경합니다. 학습 데이터는 공식 파이프라인으로만 생성합니다.
RUN_PIPELINE = False
RUN_SMOKE = False
if RUN_PIPELINE:
    subprocess.run(pipeline_command, cwd=PROJECT_ROOT, check=True)
if RUN_SMOKE:
    if get_device() == "cpu":
        raise RuntimeError("로컬 GPU가 감지되지 않았습니다. 커널과 PyTorch 설치를 확인하세요.")
    if "MuSGD" not in inspect.getsource(BaseTrainer.build_optimizer):
        raise RuntimeError("설치된 ultralytics가 MuSGD를 지원하지 않습니다.")
    subprocess.run(
        train_command, cwd=PROJECT_ROOT, env={**os.environ, "WANDB_MODE": "disabled"}, check=True
    )